# TiO2 metalens unit cell at 866 nm: period sweep (P, r, h)

Single TiO2 cylindrical meta-atom on SiO2/Si substrate, periodic in x/y, plane wave from below. Records the complex (0,0)-order transmission per geometry to build the phase library r(phi). This is the TiO2 leg of the ITO/Si3N4/TiO2 material comparison.

Sweep grid: 3 periods x 8 radii (adaptive with P) x 4 heights = 96 sims, about 2.4 FC. Result: TiO2 (n=2.4) reaches a full 1.06x2pi span at P=500 nm, h=800 nm with mean |t| ~0.97, the platform used by every design.

In [ ]:
# --- repo-root bootstrap: makes data/ paths resolve from any notebook location ---
import os
from pathlib import Path
_root = Path.cwd()
while _root != _root.parent and not (_root / '.git').is_dir():
    _root = _root.parent
if (_root / '.git').is_dir():
    os.chdir(_root)


In [8]:
import numpy as np
import matplotlib.pyplot as plt
import tidy3d as td
import tidy3d.web as web

## Parameters

Stack (bottom → top, z in µm): Si substrate (semi-infinite) → SiO2 cladding 0-5 → plane wave source @ z=5 → SiO2 cladding 5-10 → TiO2 cylindrical tooth on top.

**Baseline period `P = 0.40 µm`** is the middle of the swept range, kept as the sweep template. The actual phase-library sweep iterates over a list of periods `P_list`, the baseline is just the middle of that range, 

`P_list` (defined later) covers {300, 400, 500} nm. For TiO2 (n ≈ 2.4), the in-material subwavelength constraint is `P · n_TiO2 / λ < 1`, i.e. `P < 361 nm`, strictly only the P=300 case is subwavelength inside the tooth itself. At P=400, 500 there are higher-order propagating modes *inside the tooth*, but because the tooth is sub-µm in height the impact on the (0,0)-order transmission is limited (this is the same regime that published TiO2 metalens designs operate in). The wave is comfortably subwavelength in air (P/λ ≤ 0.58) and SiO2 cladding (P·n_SiO2/λ ≤ 0.84). `pol_angle` toggles linear polarization, set to 0 for E_x or `np.pi/2` for E_y.

In [9]:
# operating point
lda0 = 0.866                       # free-space wavelength (µm)
freq0 = td.C_0 / lda0
fwidth = freq0 / 10.0

# unit cell - BASELINE period for the single-sim viz / cost cells. The
# phase-library sweep below overrides this per simulation via P_list.
# 0.40 µm sits at the middle of the swept range (300-500 nm).
P = 0.40                           # baseline period (µm) - for single-sim viz only

# meta-atom (baseline for single-sim viz, the sweep overrides r and h)
r_tooth = 0.15                     # TiO2 cylinder radius (µm)
h_tooth = 0.40                     # TiO2 cylinder height (µm)

# cladding (user spec: 5 µm below source + 5 µm above source = 10 µm total SiO2)
t_clad_lower = 5.0
t_clad_upper = 5.0

# vertical layout
z_sub_top = 0.0                                    # Si / SiO2 interface
z_source = z_sub_top + t_clad_lower                # 5.0 µm: plane wave plane
z_clad_top = z_source + t_clad_upper               # 10.0 µm: SiO2 / air interface (base of tooth)
z_tooth_top = z_clad_top + h_tooth

# z-buffer between structures/monitors and PML. Default PML is 12 layers
# (~0.5·λ thick at 25 cells/wvl). Tidy3D's rule of thumb is ≥ 0.5·λ
# between any structure or monitor and the PML *inner* edge.
buffer_top = 2.0 * lda0
buffer_bot = 1.0 * lda0
z_min_sim = z_sub_top - buffer_bot
z_max_sim = z_tooth_top + buffer_top
Lz = z_max_sim - z_min_sim
z_center = 0.5 * (z_min_sim + z_max_sim)

# polarization (Ex by default, swap to np.pi/2 for Ey and re-run)
pol_angle = 0.0

# effective infinity for substrate / cladding boxes
inf_eff = 1e3

## Materials

- **Si substrate**, `cSi/Palik_Lossy` (Palik, 0.1-1.4 µm).
- **SiO2 cladding**, `SiO2/Palik_Lossless` (Palik, 0.15-5 µm).
- **TiO2 meta-atom**, non-dispersive `td.Medium(permittivity=2.4²)`. Conservative literature value for amorphous / ALD TiO2 at 866 nm: anatase phase gives ~2.45 and rutile ~2.65, so n = 2.40 is the floor, real fabricated material is likely to perform somewhat better. Lossless at this wavelength. Single-frequency design, revisit dispersion only for the post-design wideband validation pass.

In [10]:
Si = td.material_library['cSi']['Palik_Lossy']
SiO2 = td.material_library['SiO2']['Palik_Lossless']

# TiO2 at 866 nm - fixed-index, lossless. n=2.4 is a conservative
# value for amorphous / ALD TiO2 at this wavelength. Anatase is
# typically ~2.45 and rutile ~2.65, so 2.4 is the floor and real
# fabricated material is likely to perform somewhat better.
n_tio2 = 2.4
TiO2 = td.Medium(permittivity=n_tio2 ** 2)

## Structures

In [11]:
substrate = td.Structure(
    geometry=td.Box.from_bounds(
        rmin=(-inf_eff, -inf_eff, -inf_eff),
        rmax=(inf_eff, inf_eff, z_sub_top),
    ),
    medium=Si,
    name='Si_substrate',
)

cladding = td.Structure(
    geometry=td.Box.from_bounds(
        rmin=(-inf_eff, -inf_eff, z_sub_top),
        rmax=(inf_eff, inf_eff, z_clad_top),
    ),
    medium=SiO2,
    name='SiO2_cladding',
)

tooth = td.Structure(
    geometry=td.Cylinder(
        center=(0, 0, z_clad_top + h_tooth / 2),
        radius=r_tooth,
        length=h_tooth,
        axis=2,
    ),
    medium=TiO2,
    name='TiO2_tooth',
)

## Source

Linearly polarized plane wave injected upward (`direction='+'`) from inside the lower SiO2 cladding.

In [12]:
source = td.PlaneWave(
    center=(0, 0, z_source),
    size=(td.inf, td.inf, 0),
    source_time=td.GaussianPulse(freq0=freq0, fwidth=fwidth),
    direction='+',
    pol_angle=pol_angle,
    name='plane_wave',
)

## Monitors

- `DiffractionMonitor` 0.5·λ above the tooth (uniform air, lossless), complex transmission amplitudes per diffraction order, the (0,0) order gives the meta-atom phase and amplitude used in metalens design.
- `FluxMonitor` above the tooth, absolute transmitted power, for sanity-checking unitarity.
- `FluxMonitor` 0.5·λ below the source (inside SiO2 cladding), reflected power. We use a flux monitor here instead of a `DiffractionMonitor` because the latter requires a strictly lossless medium, and `SiO2/Palik_Lossless` has a residual ε″ ~ 3e-11 that the validator rejects. Flux gives us reflectance, which is what we need.
- `FieldMonitor` on the x-z plane through the cylinder axis, visualize standing waves and confirm the wave is propagating cleanly through the cladding.

In [13]:
mon_t = td.DiffractionMonitor(
    center=(0, 0, z_tooth_top + 0.5 * lda0),
    size=(td.inf, td.inf, 0),
    freqs=[freq0],
    normal_dir='+',
    name='t_orders',
)

mon_flux_t = td.FluxMonitor(
    center=(0, 0, z_tooth_top + 0.5 * lda0),
    size=(td.inf, td.inf, 0),
    freqs=[freq0],
    name='flux_t',
)

mon_flux_r = td.FluxMonitor(
    center=(0, 0, z_source - 0.5 * lda0),
    size=(td.inf, td.inf, 0),
    freqs=[freq0],
    name='flux_r',
)

mon_field_xz = td.FieldMonitor(
    center=(0, 0, z_center),
    size=(td.inf, 0, td.inf),
    freqs=[freq0],
    name='field_xz',
)

## Simulation

Periodic boundaries in x/y enforce the unit-cell lattice. PML on ±z absorbs reflections back into Si (which is itself lossy at 866 nm) and transmission into the air half-space.

In [14]:
boundary_spec = td.BoundarySpec(
    x=td.Boundary.periodic(),
    y=td.Boundary.periodic(),
    z=td.Boundary(minus=td.PML(), plus=td.PML()),
)

sim = td.Simulation(
    center=(0, 0, z_center),
    size=(P, P, Lz),
    grid_spec=td.GridSpec.auto(min_steps_per_wvl=25, wavelength=lda0),
    structures=[substrate, cladding, tooth],
    sources=[source],
    monitors=[mon_t, mon_flux_t, mon_flux_r, mon_field_xz],
    run_time=td.RunTimeSpec(quality_factor=1),
    boundary_spec=boundary_spec,
)

print(f"grid cells: {sim.grid.num_cells}  (total ≈ {int(np.prod(sim.grid.num_cells)/1e3)}k)")
print(f"size (µm): {sim.size}")

grid cells: [44, 44, 618]  (total ≈ 1196k)
size (µm): (0.4, 0.4, 12.998)


# Phase library sweep: (period, radius, height) grid

Sweep TiO2 tooth radius and height across periods P = {300, 400, 500} nm, recording complex (0,0)-order transmission. Radii span 40 nm to P/2-20 nm in 8 steps (keeps a ~40 nm gap at max radius). Heights h = {200, 400, 600, 800} nm. Total 96 sims, about 2.4 FC. The submission cell is commented out, review batch.estimate_cost() first.

In [17]:
def make_sim(P_var, r, h):
    """Build a unit-cell Simulation for a given TiO2 (period, radius, height) in µm.

    Each (P, r, h) sim rebuilds the lattice period (sim x/y size), the
    tooth, and repositions the top monitors. Sim z-extent grows with h
    to keep ≥ 0.5·λ PML clearance for the top monitors. The lower-half
    geometry - substrate, cladding, source, reflection monitor - is
    invariant. FieldMonitor is dropped from the sweep (costs storage).
    """
    z_tooth_top_h = z_clad_top + h
    z_max_h = z_tooth_top_h + buffer_top
    Lz_h = z_max_h - z_min_sim
    z_center_h = 0.5 * (z_min_sim + z_max_h)

    tooth_var = td.Structure(
        geometry=td.Cylinder(
            center=(0, 0, z_clad_top + h / 2),
            radius=r,
            length=h,
            axis=2,
        ),
        medium=TiO2,
        name='TiO2_tooth',
    )
    mon_t_h = mon_t.updated_copy(center=(0, 0, z_tooth_top_h + 0.5 * lda0))
    mon_flux_t_h = mon_flux_t.updated_copy(center=(0, 0, z_tooth_top_h + 0.5 * lda0))

    return sim.updated_copy(
        center=(0, 0, z_center_h),
        size=(P_var, P_var, Lz_h),
        structures=[substrate, cladding, tooth_var],
        monitors=[mon_t_h, mon_flux_t_h, mon_flux_r],
    )


# co-polarized component of the DiffractionMonitor amplitudes:
#   pol_angle = 0  (Ex) → 'p' (E in x-z plane, since angle_phi = 0)
#   pol_angle = π/2 (Ey) → 's' (E perpendicular to x-z plane)
co_pol = 'p' if np.isclose(pol_angle, 0.0) else 's'

# 3D sweep grid: period × radius × height. The radius range scales with
# P (we require r < P/2 with a small inter-tooth gap), so radii are
# stored per-period. Grid held identical to the ITO and Si3N4 sweepP
# notebooks for apples-to-apples comparison.
P_list = np.array([0.30, 0.40, 0.50])             # tooth periods (µm)
h_list = np.array([0.20, 0.40, 0.60, 0.80])       # tooth heights (µm)
n_r    = 8                                         # radii per period

def r_list_for_P(P_val, n=n_r, gap=0.02):
    """Adaptive radius list scaling with P. 40 nm minimum, P/2 - gap maximum."""
    return np.linspace(0.04, P_val / 2 - gap, n)

total_sims = len(P_list) * n_r * len(h_list)
print(f"periods:        {len(P_list)} values  P = {(P_list*1e3).astype(int)} nm")
print(f"radii per P:    {n_r}  (linspace from 40 nm to P/2 - 20 nm)")
print(f"heights:        {len(h_list)} values  h = {(h_list*1e3).astype(int)} nm")
print(f"total sims:     {total_sims}  (each ≈ 0.025 FC → ~{total_sims*0.025:.2f} FC)")
for P_val in P_list:
    r_vals = r_list_for_P(P_val)
    ar_max = h_list[-1] / (2 * r_vals[0])
    print(f"  P = {P_val*1e3:.0f} nm:  r = {r_vals[0]*1e3:.0f}-{r_vals[-1]*1e3:.0f} nm  "
          f"(gap at r_max = {(P_val - 2*r_vals[-1])*1e3:.0f} nm, AR_max = {ar_max:.1f}:1)")
print(f"co-polarized DiffractionMonitor channel: '{co_pol}'")

periods:        3 values  P = [300 400 500] nm
radii per P:    8  (linspace from 40 nm to P/2 - 20 nm)
heights:        4 values  h = [200 400 600 800] nm
total sims:     96  (each ≈ 0.025 FC → ~2.40 FC)
  P = 300 nm:  r = 40-130 nm  (gap at r_max = 40 nm, AR_max = 10.0:1)
  P = 400 nm:  r = 40-180 nm  (gap at r_max = 40 nm, AR_max = 10.0:1)
  P = 500 nm:  r = 40-230 nm  (gap at r_max = 40 nm, AR_max = 10.0:1)
co-polarized DiffractionMonitor channel: 'p'


## Build batch and estimate cost

`web.Batch` runs the simulations in parallel on the cloud. `batch.estimate_cost()` is the safe gate before submitting, never call `batch.run()` without checking this first.

In [18]:
sims = {f"tio2_sweepP_P_{P_val:.3f}_r_{r:.4f}_h_{h:.4f}": make_sim(P_val, r, h)
        for P_val in P_list
        for r in r_list_for_P(P_val)
        for h in h_list}

batch = web.Batch(simulations=sims, verbose=True)
batch_cost = batch.estimate_cost()
print(f"estimated max FlexCredit cost for the batch: {batch_cost}")

16:14:23 Pacific Daylight Time No Flexcredit cost for batch as all simulations  
                               were restored from local cache.

estimated max FlexCredit cost for the batch: 0.0


## Run the batch, uncomment after reviewing the cost estimate above

Results stream to `data/tio2_sweepP_P_*_r_*_h_*.hdf5` (the `tio2_sweepP_` prefix separates these files from the ITO and Si3N4 sweepP result files). Pull each one by `batch_results[f"tio2_sweepP_P_{P:.3f}_r_{r:.4f}_h_{h:.4f}"]` after the run completes.

In [14]:
batch_results = batch.run(path_dir='data')

Output()

17:58:40 Pacific Daylight Time Started working on Batch containing 96 tasks.

17:59:58 Pacific Daylight Time Maximum FlexCredit cost: 2.425 for the whole     
                               batch.

                               Use 'Batch.real_cost()' to get the billed        
                               FlexCredit cost after completion.

Output()

18:00:19 Pacific Daylight Time Batch complete.

## Extract complex transmission and plot the phase library

For each (P, r, h), pull the (0,0)-order co-polarized amplitude from `t_orders.amps`. Phase is unwrapped along the radius axis at each (P, h) column and rezeroed to the smallest radius.

Plot layout: one **row per period** × three columns (phase library, amplitude, loss). Within each row, one curve per height (color-coded by `h`). The dashed gray line in each phase panel marks the 1·2π target.

The summary print at the end gives the max phase span and mean amplitude for every (P, h) combination, the headline numbers for comparing periods.

In [ ]:
# Extract the (0,0)-order transmission from the batch and persist the sweep dataset.
import os
n_P, n_h_loc = len(P_list), len(h_list)
data_per_P = {}
for P_val in P_list:
    r_vals = r_list_for_P(P_val)
    t      = np.zeros((len(r_vals), n_h_loc), dtype=complex)
    T_flux = np.zeros((len(r_vals), n_h_loc))
    R_flux = np.zeros((len(r_vals), n_h_loc))
    for i, r in enumerate(r_vals):
        for j, h in enumerate(h_list):
            sd = batch_results[f"tio2_sweepP_P_{P_val:.3f}_r_{r:.4f}_h_{h:.4f}"]
            amp_00 = sd['t_orders'].amps.sel(f=freq0, polarization=co_pol, orders_x=0, orders_y=0)
            t[i, j]      = complex(amp_00.values)
            T_flux[i, j] = float(sd['flux_t'].flux.sel(f=freq0).values)
            R_flux[i, j] = float(np.abs(sd['flux_r'].flux.sel(f=freq0).values))
    phase = np.unwrap(np.angle(t), axis=0)
    phase -= phase[0:1, :]
    data_per_P[float(P_val)] = dict(r=r_vals, t=t, phase=phase, T=T_flux, R=R_flux)
os.makedirs('data', exist_ok=True)
np.savez('data/sweep_data_tio2.npz', data_per_P=data_per_P)
print('extracted and saved -> data/sweep_data_tio2.npz')

# === Plot: rows = P, cols = (phase, amplitude, loss), curves colored by h ===
fig, axes = plt.subplots(n_P, 3, figsize=(15, 4.5 * n_P), tight_layout=True)
if n_P == 1:
    axes = axes[np.newaxis, :]
colors = plt.cm.viridis(np.linspace(0, 0.9, n_h_loc))

for k, P_val in enumerate(P_list):
    d = data_per_P[float(P_val)]
    amplitude = np.abs(d['t'])
    loss = 1.0 - d['T'] - d['R']
    for j, h in enumerate(h_list):
        label = f"h = {h*1e3:.0f} nm"
        axes[k, 0].plot(d['r'] * 1e3, d['phase'][:, j] / (2 * np.pi), '-o', color=colors[j], label=label)
        axes[k, 1].plot(d['r'] * 1e3, amplitude[:, j],                '-o', color=colors[j], label=label)
        axes[k, 2].plot(d['r'] * 1e3, loss[:, j],                     '-o', color=colors[j], label=label)
    axes[k, 0].set(xlabel='tooth radius (nm)', ylabel=r'transmission phase  ($2\pi$)',
                    title=f'phase library - P = {P_val*1e3:.0f} nm')
    axes[k, 0].axhline(1.0, ls='--', color='gray', lw=0.8)
    axes[k, 0].grid(alpha=0.3); axes[k, 0].legend(fontsize=8)
    axes[k, 1].set(xlabel='tooth radius (nm)', ylabel=r'|$t_{00}$|',
                    title=f'co-pol amplitude - P = {P_val*1e3:.0f} nm', ylim=(0, 1.1))
    axes[k, 1].grid(alpha=0.3); axes[k, 1].legend(fontsize=8)
    axes[k, 2].set(xlabel='tooth radius (nm)', ylabel='1 - T - R',
                    title=f'loss - P = {P_val*1e3:.0f} nm')
    axes[k, 2].grid(alpha=0.3); axes[k, 2].legend(fontsize=8)

plt.show()

print("\nphase span across radii, per (P, h):")
for P_val in P_list:
    d = data_per_P[float(P_val)]
    print(f"  P = {P_val*1e3:.0f} nm:")
    for j, h in enumerate(h_list):
        span = d['phase'][-1, j] - d['phase'][0, j]
        mean_amp = np.abs(d['t'][:, j]).mean()
        print(f"    h = {h*1e3:>4.0f} nm:  {span:5.2f} rad  ({span/(2*np.pi):.2f}x2pi)  "
              f"mean |t| = {mean_amp:.2f}")


In [ ]:
# Phase library heatmap (span across P and h) and the best cell.
n_P_loc, n_h_loc = len(P_list), len(h_list)
span_grid = np.zeros((n_h_loc, n_P_loc))
amp_grid  = np.zeros_like(span_grid)
for k, P_val in enumerate(P_list):
    d = data_per_P[float(P_val)]
    for j, h in enumerate(h_list):
        span_grid[j, k] = (d['phase'][-1, j] - d['phase'][0, j]) / (2 * np.pi)
        amp_grid[j, k]  = np.abs(d['t'][:, j]).mean()

j_best, k_best = np.unravel_index(np.argmax(span_grid), span_grid.shape)
P_best = float(P_list[k_best])
h_best = float(h_list[j_best])
span_best = float(span_grid[j_best, k_best])

fig, (ax_hm, ax_lib) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)

hm_extent = [P_list[0]*1e3 - 50,  P_list[-1]*1e3 + 50,
             h_list[0]*1e3 - 100, h_list[-1]*1e3 + 100]
im = ax_hm.imshow(span_grid, aspect='auto', origin='lower', cmap='viridis',
                  vmin=0, vmax=max(1.0, span_grid.max()), extent=hm_extent)
for j, h in enumerate(h_list):
    for k, P_val in enumerate(P_list):
        val = span_grid[j, k]
        ax_hm.text(P_val*1e3, h*1e3, f'{val:.2f}',
                   ha='center', va='center',
                   color='white' if val < 0.7 else 'black',
                   fontsize=10, fontweight='bold')
ax_hm.plot(P_best*1e3, h_best*1e3, marker='*', markersize=22,
           color='red', markeredgecolor='white', markeredgewidth=1.5,
           label=f'best: {span_best:.2f}x2pi')
if span_grid.max() >= 1.0:
    ax_hm.contour(P_list*1e3, h_list*1e3, span_grid, levels=[1.0],
                  colors='red', linewidths=2.5)
cbar = plt.colorbar(im, ax=ax_hm)
cbar.set_label('phase span / 2pi')
ax_hm.set(xlabel='period P (nm)', ylabel='tooth height h (nm)',
          title='TiO2 phase library coverage  (max span across r, per (P, h))')
ax_hm.legend(loc='lower right')

d = data_per_P[P_best]
ax_lib.plot(d['r']*1e3, d['phase'][:, j_best] / (2 * np.pi),
            '-o', color='C0', lw=2.5, label='realized phase (x 2pi)')
ax_lib.plot(d['r']*1e3, np.abs(d['t'][:, j_best]),
            '-s', color='C1', lw=2.5, label=r'realized amplitude  $|t_{00}|$')
ax_lib.axhline(1.0, ls='--', color='red', lw=1.5, label='1x2pi target')
ax_lib.fill_between([d['r'][0]*1e3, d['r'][-1]*1e3],
                    span_best, 1.0, color='red', alpha=0.1,
                    label=f'deficit: {(1 - span_best):.2f}x2pi')
ax_lib.set(xlabel='tooth radius (nm)', ylabel='value',
           title=f'best TiO2 library:  P={P_best*1e3:.0f} nm,  h={h_best*1e3:.0f} nm',
           ylim=(0, max(1.15, span_best*1.1)))
ax_lib.grid(alpha=0.3); ax_lib.legend(loc='center right', fontsize=9)

plt.show()

print(f"BEST CELL:  P = {P_best*1e3:.0f} nm,  h = {h_best*1e3:.0f} nm")
print(f"  phase span:      {span_best:.3f}x2pi   ({span_best*100:.0f}% of full 2pi)")
print(f"  mean amplitude:  {amp_grid[j_best, k_best]:.3f}")
print(f"  radius sweep:    {d['r'][0]*1e3:.0f}-{d['r'][-1]*1e3:.0f} nm  "
      f"({len(d['r'])} unique values)")
if span_best < 0.95:
    deficit_rad = (1.0 - span_best) * 2 * np.pi
    slope_rad_per_um = (span_best * 2 * np.pi) / h_best
    h_for_2pi = (2 * np.pi) / slope_rad_per_um
    print(f"  deficit:         {deficit_rad:.2f} rad to full 2pi")
    print(f"  -> linear extrapolation needs h ~ {h_for_2pi:.2f} um at this P for full 2pi")
else:
    print(f"  -> library spans full 2pi - usable for lens layout.")


# Phase library: TiO2 at P=500 nm, h=800 nm

The (P=500, h=800) cell reaches 1.06x2pi with mean |t| ~0.97, a monotonic full-2pi library. This cell promotes it to a design artifact. It builds cubic-spline interpolants phi_of_r, r_of_phi (used by the layout code), and amp_of_r, saves data/phase_library_tio2.npz with the sample points and metadata, and draws a diagnostic figure of the forward library, amplitude, and inverse lookup.

In [ ]:
from scipy.interpolate import CubicSpline
import os

P_chosen = 0.500
h_chosen = 0.800
material_label = 'TiO2'
artifact_path = 'data/phase_library_tio2.npz'
d = data_per_P[float(P_chosen)]
j = int(np.argmin(np.abs(h_list - h_chosen)))
r_data   = d['r']
phi_data = d['phase'][:, j]
t_data   = d['t'][:, j]
amp_data = np.abs(t_data)

span_2pi      = (phi_data[-1] - phi_data[0]) / (2 * np.pi)
is_monotonic  = np.all(np.diff(phi_data) > 0)
amp_mean      = amp_data.mean()
amp_min       = amp_data.min()

print(f"\n=== Phase library check ({material_label}) ===")
print(f"  P:                  {P_chosen*1e3:.0f} nm")
print(f"  h:                  {h_chosen*1e3:.0f} nm")
print(f"  r range:            {r_data[0]*1e3:.0f}-{r_data[-1]*1e3:.0f} nm  ({len(r_data)} sample points)")
print(f"  phase span:         {span_2pi:.3f}x2pi")
print(f"  monotonic in r:     {is_monotonic}")
print(f"  amp mean / min:     {amp_mean:.3f} / {amp_min:.3f}")

assert is_monotonic, "Phase must be monotonic in r to invert the library."
if span_2pi < 1.0:
    print(f"  WARNING: span < 2pi -> library will need extrapolation for some target phases.")
else:
    print(f"  -> full 2pi coverage; library is usable as-is for lens layout.")

phi_of_r = CubicSpline(r_data,  phi_data)
amp_of_r = CubicSpline(r_data,  amp_data)
r_of_phi = CubicSpline(phi_data, r_data)

r_probe  = np.linspace(r_data[0]*1.01, r_data[-1]*0.99, 50)
phi_probe = phi_of_r(r_probe)
r_back    = r_of_phi(phi_probe)
roundtrip_err_nm = float(np.max(np.abs(r_probe - r_back))) * 1e3
print(f"  round-trip max err: {roundtrip_err_nm:.3f} nm")

os.makedirs('data', exist_ok=True)
np.savez(artifact_path,
    material      = material_label,
    n_material    = float(n_tio2),
    wavelength_um = float(lda0),
    period_um     = float(P_chosen),
    height_um     = float(h_chosen),
    r_um          = r_data,
    phase_rad     = phi_data,
    amplitude     = amp_data,
    complex_t     = t_data,
    sweep_source  = 'metalens_unit_cell_tio2_sweepP.ipynb',
)
print(f"\nlibrary saved -> {artifact_path}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), tight_layout=True)
r_fine   = np.linspace(r_data[0],   r_data[-1],   200)
phi_fine = np.linspace(phi_data[0], phi_data[-1], 200)

phi_demo = 2 * np.pi * np.array([0.0, 0.25, 0.5, 0.75, 0.95])
r_demo   = r_of_phi(phi_demo)

ax = axes[0]
ax.plot(r_fine*1e3, phi_of_r(r_fine)/(2*np.pi), '-', color='C0', lw=2, label='cubic spline')
ax.plot(r_data*1e3, phi_data/(2*np.pi),         'o', color='C0', ms=9, mfc='white', mew=2, label='FDTD samples')
ax.axhline(1.0, ls='--', color='#c0392b', lw=1.2, label='1x2pi target')
for phi_t, r_t in zip(phi_demo, r_demo):
    ax.plot([r_t*1e3, r_t*1e3], [0, phi_t/(2*np.pi)], ls=':', color='gray', lw=0.7)
    ax.plot(r_t*1e3, phi_t/(2*np.pi), '*', color='red', ms=14, mec='black', mew=0.5, zorder=5)
ax.set(xlabel='tooth radius  r (nm)', ylabel=r'phase  $\varphi / 2\pi$',
       title=r'forward library  $\varphi(r)$', ylim=(-0.05, max(1.15, span_2pi*1.05)))
ax.grid(alpha=0.3); ax.legend(fontsize=9, loc='upper left')

ax = axes[1]
ax.plot(r_fine*1e3, amp_of_r(r_fine), '-', color='C1', lw=2, label='cubic spline')
ax.plot(r_data*1e3, amp_data,         's', color='C1', ms=9, mfc='white', mew=2, label='FDTD samples')
ax.axhline(0.7, ls=':', color='gray', lw=0.8, label='0.7 floor')
ax.axhline(amp_mean, ls='-', color='gray', lw=0.5, alpha=0.4, label=f'mean = {amp_mean:.2f}')
ax.set(xlabel='tooth radius  r (nm)', ylabel=r'$|t_{00}|$',
       title='amplitude library  |t|(r)', ylim=(0, 1.15))
ax.grid(alpha=0.3); ax.legend(fontsize=9, loc='lower left')

ax = axes[2]
ax.plot(phi_fine/(2*np.pi), r_of_phi(phi_fine)*1e3, '-', color='C2', lw=2, label='cubic spline')
ax.plot(phi_data/(2*np.pi), r_data*1e3,             'D', color='C2', ms=8, mfc='white', mew=2, label='FDTD samples')
for phi_t, r_t in zip(phi_demo, r_demo):
    ax.plot([phi_t/(2*np.pi), phi_t/(2*np.pi)], [r_data[0]*1e3, r_t*1e3], ls=':', color='gray', lw=0.7)
    ax.plot(phi_t/(2*np.pi), r_t*1e3, '*', color='red', ms=14, mec='black', mew=0.5, zorder=5)
ax.set(xlabel=r'target phase  $\varphi / 2\pi$', ylabel='required radius  r (nm)',
       title=r'inverse library  $r(\varphi)$  - used in lens layout')
ax.grid(alpha=0.3); ax.legend(fontsize=9, loc='lower right')

plt.suptitle(
    f'{material_label} phase library at 866 nm:  '
    f'P = {P_chosen*1e3:.0f} nm,  h = {h_chosen*1e3:.0f} nm,  '
    f'span = {span_2pi:.2f}x2pi,  mean |t| = {amp_mean:.2f}',
    fontsize=12, fontweight='bold', y=1.02)
plt.show()

print(f"\n=== Inverse-library lookup table  (TiO2, P={P_chosen*1e3:.0f} nm, h={h_chosen*1e3:.0f} nm) ===")
print(f"  {'target phi':>14s}  ->  {'radius':>10s}    {'|t|':>6s}")
print(f"  {'-'*14}     {'-'*10}    {'-'*6}")
for phi_t in np.linspace(0, min(2*np.pi, span_2pi*2*np.pi), 9):
    r_req     = float(r_of_phi(phi_t))
    amp_at_r  = float(amp_of_r(r_req))
    print(f"  {phi_t/(2*np.pi):5.3f}x2pi     ->   {r_req*1e3:6.1f} nm    {amp_at_r:.3f}")


# Sweep summary: TiO2 (P, r, h)

Grid: P {300, 400, 500} nm x h {200, 400, 600, 800} nm x 8 radii per P = 96 sims, about 2.4 FC.

Result: phase span grows with period and height, as (n-1)*k*h*fill. The (P=500, h=800) cell reaches 1.06x2pi with mean |t| ~0.97, the first cell past the full-2pi target and the locked platform for all designs. Lower cells fall short (0.66x2pi at P=500, h=600).

Material comparison: at matched (P, h), spans scale roughly with (n-1), so TiO2 (n=2.4) exceeds Si3N4 (n=2.0) and ITO. The three-material figure is in ../analysis/metalens_phase_library_comparison.ipynb. Note the ITO leg there used a lossless placeholder index. Real ITO is n ~1.90 with absorption (docs/THEORY.md section 8).